In [2]:
!pip install numpy pandas scikit-learn matplotlib seaborn -q

In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [2]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1500

regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
amount = (prices * quantities).astype(float)

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(35, 9, n).round().astype(int),
    "region": regions,
    "membership": membership,
    "channel": channels,
    "category": categories,
    "price": prices.astype(float),
    "quantity": quantities,
    "amount": amount,
})

# 오염 심기: 결측·이상치·표기 혼재
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
orders.loc[8, "customer_age"] = -3              # 불가능한 음수
orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
orders.loc[21, "region"] = "Seoul"              # 영문 표기
orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()

이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [3]:
# ─────────────────────────────────────────────
# 시나리오 0 — 원본 CSV 파일 준비 (실무에서는 외부에서 받아오는 단계)
# ─────────────────────────────────────────────

work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv  = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"

orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")

원본 CSV 저장: d006_work\orders_raw.csv — 81901 bytes


In [4]:
# 시나리오 1 — 단계 함수들
def _log(step_name, before_shape, after_shape, verbose, extra=""):
    if not verbose:
        return
    removed = before_shape[0] - after_shape[0]
    print(f"  [{step_name}] {before_shape} → {after_shape}  "
          f"(행 {removed:+d}){(' | ' + extra) if extra else ''}")

# 각 단계별 로그를 옵션으로 출력할 수 있도록 _log() 함수를 정의. 
# verbose=True로 설정하면 각 단계별 행 수 변화를 확인할 수 있음

def load_orders(path, verbose=False):
    df = pd.read_csv(path)
    if verbose:
        print(f"  [load_orders] 로드 완료: {df.shape}")
    return df


def clean_strings_full(df, verbose=False):
    before = df.shape
    out = df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )
    _log("clean_strings_full", before, out.shape, verbose, "문자열 정리(행 수 불변)")
    return out


def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10, verbose=False):
    before = df.shape[0]

    step_na = df.dropna(subset=["amount", "customer_age"])
    n_na = before - step_na.shape[0]

    step_age = step_na.query("@age_min < customer_age < @age_max")
    n_age = step_na.shape[0] - step_age.shape[0]

    step_qty = step_age.query("quantity <= @qty_max")
    n_qty = step_age.shape[0] - step_qty.shape[0]

    step_dup = step_qty.drop_duplicates()
    n_dup = step_qty.shape[0] - step_dup.shape[0]

    out = step_dup.reset_index(drop=True)

    if verbose:
        print(f"  [drop_invalid_rows] 시작 행 수: {before}")
        print(f"    - 결측치(amount/customer_age) 제거: -{n_na}행 → {step_na.shape[0]}")
        print(f"    - 나이 범위({age_min}~{age_max}) 밖 제거: -{n_age}행 → {step_age.shape[0]}")
        print(f"    - quantity > {qty_max} 제거: -{n_qty}행 → {step_qty.shape[0]}")
        print(f"    - 중복행 제거: -{n_dup}행 → {step_dup.shape[0]}")
        print(f"    => 최종: {before} → {out.shape[0]} "
              f"(총 {before - out.shape[0]}행 제거, {100*(before-out.shape[0])/before:.1f}%)")
    return out


def add_derived(df, verbose=False):
    before = df.shape
    out = df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )
    _log("add_derived", before, out.shape, verbose, "파생 컬럼 3개 추가(행 수 불변)")
    return out


def encode_full(df, verbose=False):
    before = df.shape
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    out = pd.concat([out] + one_hots, axis=1)
    _log("encode_full", before, out.shape, verbose,
         f"인코딩 컬럼 {out.shape[1]-before[1]}개 추가(행 수 불변)")
    return out


def scale_with_robust(df, cols=("customer_age", "amount", "quantity"), verbose=False):
    before = df.shape
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    out = pd.concat([df, scaled_df], axis=1)
    _log("scale_with_robust", before, out.shape, verbose,
         f"스케일 컬럼 {len(cols)}개 추가(행 수 불변)")
    return out

print("단계 함수 6개 정의 완료 (verbose 로그 내장). 다음 셀에서 조립합니다.")

단계 함수 6개 정의 완료 (verbose 로그 내장). 다음 셀에서 조립합니다.


In [5]:
# 시나리오 2 — end-to-end 파이프라인

# verbose=True로 설정하면 각 단계별 행 수 변화를 확인할 수 있음
def preprocess(input_path, verbose=False, age_min=10, age_max=80, qty_max=10):
    if verbose:
        print(f"=== preprocess 시작: {input_path} ===")

    df = load_orders(input_path, verbose=verbose)
    df = clean_strings_full(df, verbose=verbose)
    df = drop_invalid_rows(df, age_min=age_min, age_max=age_max, qty_max=qty_max, verbose=verbose)
    df = add_derived(df, verbose=verbose)
    df = encode_full(df, verbose=verbose)
    df = scale_with_robust(df, verbose=verbose)

    if verbose:
        print(f"=== preprocess 완료: 최종 shape {df.shape} ===\n")
    return df


# 진짜 한 줄 호출 (verbose=True로 단계별 로그 확인)
clean_df = preprocess(input_csv, verbose=True)
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
clean_df.head(3)

=== preprocess 시작: d006_work\orders_raw.csv ===
  [load_orders] 로드 완료: (1500, 9)
  [clean_strings_full] (1500, 9) → (1500, 9)  (행 +0) | 문자열 정리(행 수 불변)
  [drop_invalid_rows] 시작 행 수: 1500
    - 결측치(amount/customer_age) 제거: -87행 → 1413
    - 나이 범위(10~80) 밖 제거: -8행 → 1405
    - quantity > 10 제거: -1행 → 1404
    - 중복행 제거: -0행 → 1404
    => 최종: 1500 → 1404 (총 96행 제거, 6.4%)
  [add_derived] (1404, 9) → (1404, 12)  (행 +0) | 파생 컬럼 3개 추가(행 수 불변)
  [encode_full] (1404, 12) → (1404, 25)  (행 +0) | 인코딩 컬럼 13개 추가(행 수 불변)
  [scale_with_robust] (1404, 25) → (1404, 28)  (행 +0) | 스케일 컬럼 3개 추가(행 수 불변)
=== preprocess 완료: 최종 shape (1404, 28) ===

입력 행 수 → 출력 행 수: 1500 → 1404
출력 컬럼 수: 28


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,...,ch_app,ch_web,cat_가전,cat_도서,cat_뷰티,cat_식품,cat_패션,customer_age_scaled,amount_scaled,quantity_scaled
0,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,...,1,0,1,0,0,0,0,-0.916667,10.141429,2.0
1,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,...,0,1,0,0,0,0,1,0.833333,-0.284286,0.0
2,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,...,1,0,0,0,0,0,1,0.500000,-0.284286,0.0


In [6]:
# 시나리오 3 — 저장 + 품질 리포트 함수
clean_df.to_csv(output_csv, index=False)
reloaded = pd.read_csv(output_csv)
print("저장 파일:", output_csv, "—", output_csv.stat().st_size, "bytes")
print("저장 직전 shape:", clean_df.shape)
print("다시 읽은  shape:", reloaded.shape)
print("동일한가?:", clean_df.shape == reloaded.shape)


def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    report = {
        "rows_before": before.shape[0],
        "rows_after":  after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),
        "cols_before": before.shape[1],
        "cols_after":  after.shape[1],
        "new_cols":    after.shape[1] - before.shape[1],
        "missing_top5_before": (
            before.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "missing_top5_after": (
            after.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        ),
        "dtypes_after": after.dtypes.value_counts().to_dict(),
    }
    return report

rep = quality_report(orders, clean_df)
for k, v in rep.items():
    print(f"{k}: {v}")

저장 파일: d006_work\orders_clean.csv — 199683 bytes
저장 직전 shape: (1404, 28)
다시 읽은  shape: (1404, 28)
동일한가?: True
rows_before: 1500
rows_after: 1404
removed_pct: 6.4
cols_before: 9
cols_after: 28
new_cols: 19
missing_top5_before: {'amount': 60, 'customer_age': 30, 'order_id': 0, 'membership': 0, 'region': 0}
missing_top5_after: {'order_id': 0, 'customer_age': 0, 'region': 0, 'membership': 0, 'channel': 0}
dtypes_after: {dtype('int64'): 15, dtype('float64'): 7, <StringDtype(storage='python', na_value=nan)>: 6}


In [7]:
# ------------------------------------------------------------
# 시나리오 6 — 같은 파이프라인, 시드 A vs 시드 B 비교
# ------------------------------------------------------------

def resample_with_seed(df, seed, frac=1.0, replace=True):
    # replace=True: 복원추출(부트스트랩) — 원본과 다른 분포/중복 조합 생성
    return df.sample(frac=frac, replace=replace, random_state=seed).reset_index(drop=True)


def run_pipeline_with_seed(base_df, seed, frac=1.0, replace=True, verbose=False):
    sampled = resample_with_seed(base_df, seed=seed, frac=frac, replace=replace)
    seed_csv = work_dir / f"orders_seed{seed}.csv"
    sampled.to_csv(seed_csv, index=False)
    result = preprocess(seed_csv, verbose=verbose)
    return result


def compare_two_seeds(base_df, seed_a, seed_b, frac=1.0, replace=True, verbose_a=False, verbose_b=False):
    print(f"### 시드 {seed_a} 실행 ###")
    result_a = run_pipeline_with_seed(base_df, seed=seed_a, frac=frac, replace=replace, verbose=verbose_a)

    print(f"\n### 시드 {seed_b} 실행 ###")
    result_b = run_pipeline_with_seed(base_df, seed=seed_b, frac=frac, replace=replace, verbose=verbose_b)

    print("\n" + "=" * 60)
    print(f"[비교] 시드 {seed_a} vs 시드 {seed_b}")
    print("=" * 60)

    # 1) 기본 shape 비교
    print(f"행 수:   {result_a.shape[0]} vs {result_b.shape[0]}  "
          f"(차이 {result_a.shape[0] - result_b.shape[0]:+d})")
    print(f"컬럼 수: {result_a.shape[1]} vs {result_b.shape[1]}")

    # 2) 수치형 컬럼 통계 비교
    numeric_cols = ["customer_age", "amount", "quantity", "amount_log"]
    stats_rows = []
    for col in numeric_cols:
        stats_rows.append({
            "column": col,
            f"mean_seed{seed_a}": round(result_a[col].mean(), 2),
            f"mean_seed{seed_b}": round(result_b[col].mean(), 2),
            "mean_diff": round(result_a[col].mean() - result_b[col].mean(), 2),
            f"std_seed{seed_a}": round(result_a[col].std(), 2),
            f"std_seed{seed_b}": round(result_b[col].std(), 2),
        })
    stats_df = pd.DataFrame(stats_rows)
    print("\n[수치형 컬럼 통계 비교]")
    print(stats_df.to_string(index=False))

    # 3) 스케일 결과(RobustScaler) 비교
    scaled_cols = ["customer_age_scaled", "amount_scaled", "quantity_scaled"]
    print("\n[스케일 컬럼(amount_scaled 등) 평균 비교]")
    for col in scaled_cols:
        print(f"  {col}: 시드{seed_a}={result_a[col].mean():.4f}  "
              f"vs 시드{seed_b}={result_b[col].mean():.4f}  "
              f"(차이 {result_a[col].mean() - result_b[col].mean():+.4f})")

    # 4) 범주형 분포 비교 (region_clean 등 → 원본 컬럼명 확인 필요, 예시로 membership 비율)
    print("\n[is_premium 비율 비교]")
    print(f"  시드 {seed_a}: {result_a['is_premium'].mean():.2%}")
    print(f"  시드 {seed_b}: {result_b['is_premium'].mean():.2%}")

    print("\n[amount_class 분포 비교]")
    dist_a = result_a["amount_class"].value_counts(normalize=True).round(3)
    dist_b = result_b["amount_class"].value_counts(normalize=True).round(3)
    dist_compare = pd.DataFrame({f"seed{seed_a}": dist_a, f"seed{seed_b}": dist_b}).fillna(0)
    print(dist_compare)

    return result_a, result_b, stats_df


# 실행: 시드 1과 시드 2 비교
result_seed1, result_seed2, stats_diff = compare_two_seeds(
    orders,
    seed_a=1,
    seed_b=2,
    frac=1.0,
    replace=True,
    verbose_a=True,   # True로 바꾸면 시드 1의 단계별 상세 로그 출력
    verbose_b=True,   # True로 바꾸면 시드 2의 단계별 상세 로그 출력
)

### 시드 1 실행 ###
=== preprocess 시작: d006_work\orders_seed1.csv ===
  [load_orders] 로드 완료: (1500, 9)
  [clean_strings_full] (1500, 9) → (1500, 9)  (행 +0) | 문자열 정리(행 수 불변)
  [drop_invalid_rows] 시작 행 수: 1500
    - 결측치(amount/customer_age) 제거: -91행 → 1409
    - 나이 범위(10~80) 밖 제거: -7행 → 1402
    - quantity > 10 제거: -1행 → 1401
    - 중복행 제거: -508행 → 893
    => 최종: 1500 → 893 (총 607행 제거, 40.5%)
  [add_derived] (893, 9) → (893, 12)  (행 +0) | 파생 컬럼 3개 추가(행 수 불변)
  [encode_full] (893, 12) → (893, 25)  (행 +0) | 인코딩 컬럼 13개 추가(행 수 불변)
  [scale_with_robust] (893, 25) → (893, 28)  (행 +0) | 스케일 컬럼 3개 추가(행 수 불변)
=== preprocess 완료: 최종 shape (893, 28) ===


### 시드 2 실행 ###
=== preprocess 시작: d006_work\orders_seed2.csv ===
  [load_orders] 로드 완료: (1500, 9)
  [clean_strings_full] (1500, 9) → (1500, 9)  (행 +0) | 문자열 정리(행 수 불변)
  [drop_invalid_rows] 시작 행 수: 1500
    - 결측치(amount/customer_age) 제거: -95행 → 1405
    - 나이 범위(10~80) 밖 제거: -9행 → 1396
    - quantity > 10 제거: -3행 → 1393
    - 중복행 제거: -485행 → 908
    => 